# wandb-init-run — faded example 2: Pass a merged config dict to wandb.init

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-init-run`. Running the beacon reports progress on the `Logging: wandb.init run` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.init run` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-init-run`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-init-run"
DD_SUBTOPIC = "Logging: wandb.init run"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

When running sweep trials, you often build a merged config dict — combining dataclass defaults with per-trial overrides — before opening the run. The merged dict is passed as `config=merged` to `wandb.init`, ensuring the dashboard shows exactly the hyperparameters used in this trial, including any per-trial overrides.

## Faded exercise 2

Implement `init_with_merged(defaults, overrides)`. Build a merged config dict (defaults first, overrides second), then call `wandb.init` with it. Return the merged dict.

Complete the blank that builds the merged dict and calls wandb.init.

**Fill in:** Build merged = {**asdict(defaults), **overrides} and call wandb.init(project=defaults.wandb_project, name=defaults.wandb_name, config=merged).

In [ ]:
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass, asdict
sys.modules.setdefault('wandb', MagicMock())
import wandb

@dataclass
class Args:
    lr: float = 1e-3
    wd: float = 0.01
    wandb_project: str = 'sweep'
    wandb_name: str = 'trial'

def init_with_merged(defaults, overrides):
    merged = {**asdict(defaults), **overrides}
    wandb.init(
        project=defaults.wandb_project,
        name=defaults.wandb_name,
        config=merged,
    )
    return merged

wandb.init.reset_mock()
args = Args()
merged = init_with_merged(args, {'lr': 1e-4})
print('merged lr:', merged.get('lr'))
print('config in init:', wandb.init.call_args.kwargs.get('config', {}).get('lr'))


import sys
from unittest.mock import MagicMock
from dataclasses import dataclass, asdict
sys.modules.setdefault('wandb', MagicMock())
import wandb

@dataclass
class Args:
    lr: float = 1e-3
    wd: float = 0.01
    wandb_project: str = 'sweep'
    wandb_name: str = 'trial'

def init_with_merged(defaults, overrides):
    merged = {**asdict(defaults), **overrides}
    wandb.init(project=defaults.wandb_project, name=defaults.wandb_name, config=merged)
    return merged

def _test():
    wandb.init.reset_mock()
    args = Args()
    original_lr = args.lr
    merged = init_with_merged(args, {'lr': 9e-5, 'wd': 0.001})
    assert merged['lr'] == 9e-5, 'override must win'
    assert merged['wd'] == 0.001
    assert args.lr == original_lr, 'defaults must not be mutated'
    kw = wandb.init.call_args.kwargs
    assert kw['config']['lr'] == 9e-5
    assert kw['project'] == 'sweep'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass, asdict
sys.modules.setdefault('wandb', MagicMock())
import wandb

@dataclass
class Args:
    lr: float = 1e-3
    wd: float = 0.01
    wandb_project: str = 'sweep'
    wandb_name: str = 'trial'

def init_with_merged(defaults, overrides):
    merged = {**asdict(defaults), **overrides}
    wandb.init(
        project=defaults.wandb_project,
        name=defaults.wandb_name,
        config=merged,
    )
    return merged

wandb.init.reset_mock()
args = Args()
merged = init_with_merged(args, {'lr': 1e-4})
print('merged lr:', merged.get('lr'))
print('config in init:', wandb.init.call_args.kwargs.get('config', {}).get('lr'))
```
</details>